## Example 1 — the installed version, and the C extension (not the pure-Python fallback) is what runs

In [ ]:
import importlib.metadata
import pybase64

assert pybase64.__version__ == importlib.metadata.version("pybase64"), pybase64.__version__
described = pybase64.get_version()
# pybase64 falls back to pure Python when its C extension is missing; the C extension must be what runs.
assert "C extension active" in described, described
assert pybase64.b64encode is pybase64._pybase64.b64encode
print("pybase64", described)

## Example 2 — encode and decode agree with the standard library, 0 bytes to 1 MiB

In [ ]:
import base64
import os
import pybase64

for size in (0, 1, 2, 3, 57, 4096, 65537, 1 << 20):
    data = os.urandom(size)
    encoded = pybase64.b64encode(data)
    assert encoded == base64.b64encode(data), size
    assert pybase64.b64decode(encoded, validate=True) == data, size
    assert pybase64.standard_b64decode(pybase64.standard_b64encode(data)) == data, size
print("8 sizes up to 1 MiB match base64.b64encode / b64decode")

## Example 3 — URL-safe and custom alphabets, and the string / bytearray variants

In [ ]:
import base64
import pybase64

data = bytes(range(256)) * 3
assert pybase64.urlsafe_b64encode(data) == base64.urlsafe_b64encode(data)
assert pybase64.urlsafe_b64decode(pybase64.urlsafe_b64encode(data)) == data
custom = pybase64.b64encode(data, altchars=b"_:")
assert custom == base64.b64encode(data, altchars=b"_:") and pybase64.b64decode(custom, altchars=b"_:", validate=True) == data
as_string = pybase64.b64encode_as_string(data)
assert isinstance(as_string, str) and as_string.encode("ascii") == base64.b64encode(data)
as_bytearray = pybase64.b64decode_as_bytearray(as_string)
assert isinstance(as_bytearray, bytearray) and bytes(as_bytearray) == data
assert pybase64.encodebytes(data) == base64.encodebytes(data)
print("url-safe, custom alphabet, str and bytearray variants agree with the standard library")

## Example 4 — invalid input is refused when validation is on

In [ ]:
import binascii
import pybase64

for bad in (b"####", b"QUJD=", b"QU JD", "QUJDé"):
    try:
        pybase64.b64decode(bad, validate=True)
        raise AssertionError(f"{bad!r} was accepted")
    except (binascii.Error, ValueError) as error:
        print(f"{bad!r} refused: {type(error).__name__}: {error}")
# Without validation, characters outside the alphabet are skipped, as in the standard library.
assert pybase64.b64decode(b"QU JD\n") == b"ABC"

## Example 5 — the `python -m pybase64` command-line program runs and reports the installed version

In [ ]:
import subprocess
import sys
import pybase64

result = subprocess.run([sys.executable, "-m", "pybase64", "--version"], capture_output=True, text=True, timeout=120)
assert result.returncode == 0 and pybase64.__version__ in result.stdout, (result.returncode, result.stdout, result.stderr[-500:])
print("python -m pybase64 --version ->", result.stdout.strip().splitlines()[0])